## Learning Objectives

By completing this lab you will be able to:
- Explain what the **Model Context Protocol** is and why it matters for enabling LLMs to act beyond natural language processing
- Create and run an enhanced **MCP server** that uses all three core capabilities:
  - **Tools** -> to perform actions (write & delete files)
  - **Resources** -> to fetch and expose data (read files, list directories)
  - **Prompts** -> to guide structured workflows (code review & documentation)
Additionally, your server will implement MCP Context to handle **logging, progress reporting, and user elicitation**
- Build a simple but extensible **MCP client** with a CLI menu to interact with your server and a [Claude](https://claude.ai/new) chatbot
- Understand how to extend MCP for your own projects


## MCP Resources

Resources expose **data** to the client — think of them as read-only endpoints. Unlike tools (which perform actions), resources are for **passively fetching information** that already exists: files, directory listings, database records, etc.

> **Rule of thumb:** If the main purpose is to *fetch* data → use a resource. If it involves logic, side effects, or remote calls with custom config → use a tool instead.

### URI System

Every resource has a unique **URI** (Uniform Resource Identifier) that the client uses to request it. The static prefix (like `file://`, `dir://`) is chosen by you, the server developer.

```python
# Static URI — always returns the same thing (directory listing)
@mcp.resource("dir://.")
async def list_files_resource() -> dict:
    ...
```

Because URIs are fixed, clients need to know them in advance — resources are "hard-coded" in that sense.

### Resource Templates (Dynamic URIs)

A **resource template** is a resource with a parameterized URI — the `{variable}` part gets filled in by the client at request time.

```python
# Dynamic URI — {file_name} is provided by the client
@mcp.resource("file:///{file_name}")
async def read_file_resource(file_name: str) -> dict:
    ...
```

| | Resource | Resource Template |
|---|---|---|
| URI | Static (`dir://.`) | Dynamic (`file:///{file_name}`) |
| Parameters | None | One or more `{param}` in URI |
| Use case | Fixed data source | Data that varies by input |


## User Elicitation

Elicitation lets the server **pause and ask the user for input** mid-execution — before it continues. It's part of MCP Context (`ctx`).

```python
result = await ctx.elicit(
    message="Please provide the subject file name and the documentation file name",
    response_type=DocumentGeneratorSchema  # Pydantic model defining the expected fields
)
file_path = result.data.file_path  # access user's response
```

**How it works:**
1. Server calls `ctx.elicit()` with a message and a Pydantic schema
2. Client receives it and renders a form based on the schema fields
3. User fills in the form and submits
4. Server receives `result.data` and continues execution

**Requires:** a special elicitation handler in the client — without it, the call will fail.

**When to use it:**
| Scenario | Example |
|---|---|
| Missing parameters | File path not provided upfront |
| Clarification | "Are you sure you want to delete this?" |
| Progressive input | Collect info step-by-step during a long task |
